In [ ]:
import pickle
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import accuracy_score

In [ ]:
# Menghubungkan Drive ke Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = '/content/drive/MyDrive/skripsi/svm_indobert/data/final_data/2026_krl_complaints_preprocessed_4.csv'
df_test = pd.read_csv(path)

In [ ]:
label_map = {
    'Keterlambatan': 0,
    'Pelayanan': 1,
    'Kepadatan': 2,
    'Keamanan': 3,
    'Kebersihan': 4
}

inverse_label_map = {v: k for k, v in label_map.items()}

# Ground truth numerik
y_true = df_test['label'].map(label_map).values

In [ ]:
BUNDLE_PATH = '/content/drive/MyDrive/skripsi/svm_indobert/model/svm_linearsvc_data_2_2.pkl'
with open(BUNDLE_PATH, 'rb') as f:
    bundle = pickle.load(f)

svm_model = bundle['model']
vectorizer   = bundle['vectorizer']

In [ ]:
X_test_processed = df_test['text_svm']
X_test_tfidf = vectorizer.transform(X_test_processed)
y_pred_svm = svm_model.predict(X_test_tfidf)

In [ ]:
# Config
SAVE_DIR   = '/content/drive/MyDrive/skripsi/svm_indobert/model/indobert_data_2_2'
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN    = 128
BATCH_SIZE = 16

# Load  Model & Tokenizer
bert_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
bert_model.to(DEVICE)
bert_model.eval()
bert_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
def predict_bert(texts):
    preds = []

    for i in range(0, len(texts), BATCH_SIZE):

        batch_texts = texts[i:i + BATCH_SIZE]

        inputs = bert_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = bert_model(**inputs)

        batch_preds = torch.argmax(outputs.logits, dim=1)

        preds.extend(batch_preds.cpu().numpy())

    return preds

y_pred_bert = predict_bert(df_test['text_bert'].tolist())

In [ ]:
# Sanity Check
print("Accuracy SVM :", accuracy_score(y_true, y_pred_svm))
print("Accuracy BERT:", accuracy_score(y_true, y_pred_bert))

Accuracy SVM : 0.7780040733197556
Accuracy BERT: 0.7556008146639511


In [ ]:
svm_correct = (y_pred_svm == y_true)
bert_correct = (np.array(y_pred_bert) == y_true)

a = np.sum(svm_correct & bert_correct)
b = np.sum(svm_correct & ~bert_correct)
c = np.sum(~svm_correct & bert_correct)
d = np.sum(~svm_correct & ~bert_correct)

print(f"a (keduanya benar)        : {a}")
print(f"b (SVM benar, BERT salah) : {b}")
print(f"c (SVM salah, BERT benar) : {c}")
print(f"d (keduanya salah)        : {d}")
print(f"Discordant Pairs (b+c)    : {b + c}")

a (keduanya benar)        : 338
b (SVM benar, BERT salah) : 44
c (SVM salah, BERT benar) : 33
d (keduanya salah)        : 76
Discordant Pairs (b+c)    : 77


In [ ]:
table = [[a, b], [c, d]]

result = mcnemar(table, exact=False, correction=True)

print(f"\nStatistik McNemar : {result.statistic:.4f}")
print(f"p-value           : {result.pvalue:.4f}")

if result.pvalue < 0.05:
    print("→ Perbedaan SIGNIFIKAN secara statistik (p < 0.05)")
else:
    print("→ Perbedaan TIDAK signifikan secara statistik (p ≥ 0.05)")


Statistik McNemar : 1.2987
p-value           : 0.2545
→ Perbedaan TIDAK signifikan secara statistik (p ≥ 0.05)


In [ ]:
# Konversi ke label teks
y_true_label = [inverse_label_map[i] for i in y_true]
y_pred_svm_label = [inverse_label_map[i] for i in y_pred_svm]
y_pred_bert_label = [inverse_label_map[i] for i in y_pred_bert]

In [ ]:
results = pd.DataFrame({
    'tweet': df_test['full_text'],

    # versi numerik
    'actual_id': y_true,
    'svm_id': y_pred_svm,
    'bert_id': y_pred_bert,

    # versi label
    'label_aktual': y_true_label,
    'prediksi_svm': y_pred_svm_label,
    'prediksi_bert': y_pred_bert_label
})

In [ ]:
errors = results[
    (results['prediksi_svm'] != results['label_aktual']) |
    (results['prediksi_bert'] != results['label_aktual'])
]
errors.to_csv('error_analysis.csv', index=False)

In [ ]:
print(f"\nTotal kasus kesalahan: {len(errors)}")
print(errors.head(20))


Total kasus kesalahan: 153
                                                tweet  actual_id  svm_id  \
6   @CommuterLine Buset baru di benerin jam 06.00 ...          0       0   
11  @CommuterLine @kamarlintang Pecat itu pegawai ...          3       1   
13  @CommuterLine min apakah ada standar/aturan ma...          3       1   
14  @CommuterLine Commuter dr kalibata menuju kota...          3       3   
21  @CommuterLine @KAI121 commuter line penataran ...          1       0   
23  @CommuterLine grgr kereta kalian kemarin telat...          0       1   
26  @CommuterLine Posisinya di area stasiun atas m...          3       1   
27  @CommuterLine Min tpi seharusnya bisa dong sat...          1       3   
30  @CommuterLine Kai cuma minta maaf maaf maaff i...          0       1   
33  @CommuterLine ini yang ditandai emang orang bo...          3       1   
38  @CommuterLine peraturan dilarang buang ludah s...          4       3   
40  @CommuterLine sungguh nggak worth it dari jam ...       